# smsdk Example 1 — Quick Start & Querying Data

End-to-end walkthrough of the most common SDK operations: setting up a client, exploring machines and their schemas, and querying cycles / parts / downtimes / raw data with the full set of query operators.

For full method signatures and parameter reference, see [docs/README.md](../docs/README.md).

**Updated:** April 2026 — consolidates the former `Quick Start.ipynb` and `Query Examples.ipynb`.

## Setup

In [1]:
from smsdk import client
from datetime import datetime
import pandas as pd
import os

In [ ]:
# Set before running this notebook.
tenant = "demo-bottling"
api_key = ""
api_secret = ""

In [3]:
cli = client.Client(tenant)
success = cli.login('apikey', key_id=api_key, secret_id=api_secret)
assert success, 'SDK login failed — check tenant / API key / secret.'

### (Optional) Select a development pipeline / workspace
By default, the production pipeline is used. To query data from an alternate pipeline, call `select_workspace_id`. The setting persists until the client is re-instantiated.

In [4]:
# cli.select_workspace_id(workspace_id='<pipeline_id>')

## Exploring machines and their schemas

Cycle data is organized by machine, and each machine has a machine type that defines its schema. The typical lookup flow is: machine type → machine → schema.

In [5]:
# List all machine types (display names)
types = cli.get_machine_type_names()
types

['Packer',
 'Filtec',
 'Line 1',
 'L1 Full Can Conveyor',
 'L1 Warmer',
 'L1 Case Conveyor',
 'Electricity',
 'L1-Depal',
 'Palletizer',
 'Filler',
 'Empty Can Conveyor',
 'Blender_1',
 'Wrapper']

In [6]:
# List machines of a specific type
machine_type = types[0]
machines = cli.get_machine_names(source_type=machine_type)
machines

['L1 - Packer #2 (MEAD 1C)',
 'L1 - Packer #3 (MEAD 1B)',
 'L1 - Packer #4 (MEAD 1A)']

In [7]:
# Get the full schema (tag metadata) for one machine
schema = cli.get_machine_schema(machines[0])
schema.head()

,name,display,unit,sight_type,type,stream_types,raw_data_field,model,formatting,annotations,ui_hidden,ui_hidden_machines,ui_hidden_facilities,machine_type,formula
0,machine__source,Machine,,categorical,string,[],,cycle,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,starttime,Cycle Start Time,,datetime,datetime,[],,cycle,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,endtime,Cycle End Time,,datetime,datetime,[],,cycle,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,production_date,Production Day,,date,datetime,[],,cycle,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,total,Cycle Time (Net),ms,continuous,int,[],,cycle,"{'duration': True, 'formatString': 'seconds'}",NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# Filter the schema to only numeric (continuous) tags
schema_numeric = cli.get_machine_schema(machines[0], types=['continuous'])
schema_numeric.head()

,name,display,unit,sight_type,type,stream_types,raw_data_field,formatting,model,annotations,ui_hidden,ui_hidden_machines,ui_hidden_facilities,machine_type,formula
0,total,Cycle Time (Net),ms,continuous,int,[],,"{'duration': True, 'formatString': 'seconds'}",cycle,NaN,NaN,NaN,NaN,NaN,NaN
1,record_time,Cycle Time (Gross),ms,continuous,int,[],,"{'duration': True, 'formatString': 'seconds'}",cycle,NaN,NaN,NaN,NaN,NaN,NaN
2,stats__CIP Status Change__val,CIP Status Change,,continuous,float,[],CIP Status Change,{'is_convertible': False},cycle,{},False,[],[],"{'id': '05ec99d0ca304f185d58bfbe', 'name': 'mt...",NaN
3,stats__Case jam at discharge__val,Case jam at discharge,,continuous,float,[],Case jam at discharge,{'is_convertible': False},cycle,{},False,[],[],"{'id': '05ec99d0ca304f185d58bfbe', 'name': 'mt...",NaN
4,stats__Deformed Cartons__val,Deformed Cartons,,continuous,float,[],Deformed Cartons,{'is_convertible': False},cycle,{},False,[],[],"{'id': '05ec99d0ca304f185d58bfbe', 'name': 'mt...",NaN


In [9]:
# Timezone for a given machine — timestamps returned by the API are UTC; use this to convert
cli.get_machine_timezone(machines[0])

'America/Los_Angeles'

In [10]:
# Look up the internal machine type name from a machine name
cli.get_type_from_machine(machines[0])

'mt_packer'

In [11]:
# Schema at the machine-type level (vs. machine level). Returns list of dicts.
type_fields = cli.get_fields_of_machine_type(cli.get_type_from_machine(machines[0]))
pd.DataFrame(type_fields).head()

,name,display_name,unit,type,data_type,stream_types,raw_data_field,model,formatting,annotations,ui_hidden,ui_hidden_machines,ui_hidden_facilities,machine_type,formula
0,machine__source,Machine,,categorical,string,[],,cycle,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,starttime,Cycle Start Time,,datetime,datetime,[],,cycle,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,endtime,Cycle End Time,,datetime,datetime,[],,cycle,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,production_date,Production Day,,date,datetime,[],,cycle,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,total,Cycle Time (Net),ms,continuous,int,[],,cycle,"{'duration': True, 'formatString': 'seconds'}",NaN,NaN,NaN,NaN,NaN,NaN


## Querying cycles

`get_cycles` returns a cycle-level DataFrame. Below we walk through the common query operators.

> Update the date ranges in the settings cell below to something that has data in your tenant.

### Query settings (edit me)

In [12]:
start_time = datetime(2023, 4, 1)
end_time = datetime(2023, 4, 2)

### Basic cycle query
- `'End Time__gte'` / `'End Time__lte'` — greater/less-than-or-equal (use `__gt` / `__lt` for strict)
- `'_order_by'` — prefix with `-` for descending

In [13]:
query = {
    'Machine': machines[0],
    'End Time__gte': start_time,
    'End Time__lte': end_time,
    '_order_by': '-End Time',
}
df = cli.get_cycles(**query)
print(f'Shape: {df.shape}')
df.head()

_limit not specified. Maximum of 5000 rows will be returned.
_only not specified.  Selecting first 50 fields.
Shape: (0, 0)


""


### Selecting specific columns with `_only`

If you don't specify `_only`, the SDK returns the first ~50 stats plus common metadata. Pass a list of display names to select. `_only='*'` returns every field (slow).

**Note:** columns that are all-null are dropped from the result; don't be surprised if you get fewer columns back than you requested.

In [14]:
# Grab a few tag display names (skip the first few internal fields)
tag_cols = cli.get_machine_schema(machines[0])['display'].to_list()[8:13]
select_cols = ['Machine', 'End Time'] + tag_cols

query = {
    'Machine': machines[0],
    'End Time__gte': start_time,
    'End Time__lte': end_time,
    '_order_by': '-End Time',
    '_only': select_cols,
}
df = cli.get_cycles(**query)
print(f'Shape: {df.shape}')
df.head()

_limit not specified. Maximum of 5000 rows will be returned.
Shape: (0, 0)


""


### Paginating with `_limit` and `_offset`
Use these together to page through large result sets.

In [15]:
query = {
    'Machine': machines[0],
    'End Time__gte': start_time,
    'End Time__lte': end_time,
    '_order_by': '-End Time',
    '_offset': 10,
    '_limit': 500,
}
df = cli.get_cycles(**query)
print(f'Shape: {df.shape}')
df.head()

_only not specified.  Selecting first 50 fields.
Shape: (0, 0)


""


### Filtering with `__in` / `__nin` (list membership)
Use `__in` when you want several values, or `__nin` to exclude them. Most common use case: pulling data across several machines of the same type.

**Tip:** don't mix machine types in one call — the returned DataFrame will be sparse and hard to read.

In [16]:
query = {
    'Machine__in': machines[:3],
    'End Time__gte': start_time,
    'End Time__lte': end_time,
    '_order_by': '-End Time',
}
df = cli.get_cycles(**query)
print(f'Shape: {df.shape} — Machine column should have up to 3 distinct values')
df.head()

_limit not specified. Maximum of 5000 rows will be returned.
_only not specified.  Selecting first 50 fields.
Shape: (0, 0) — Machine column should have up to 3 distinct values


""


### Filtering with `__exists` (null / not-null)
Common for sparse inspection / defect fields. Pass `True` to require the field, `False` to require it be missing.

In [17]:
# Replace 'Alarms' below with a real sparse field from your tenant's schema
query = {
    'Machine': machines[0],
    'production_date__exists': False,
    'End Time__gte': start_time,
    'End Time__lte': end_time,
    '_order_by': '-End Time',
    '_only': ['Machine', 'End Time', 'production_date'],
}
df = cli.get_cycles(**query)
print(f'Shape: {df.shape}')
df.head()

_limit not specified. Maximum of 5000 rows will be returned.
Shape: (0, 0)


""


### Inequality with `__ne`

In [18]:
# Replace 'output' with a real numeric tag from your tenant's schema
query = {
    'Machine__in': machines[:3],
    'output__ne': 0,
    'End Time__gte': start_time,
    'End Time__lte': end_time,
    '_order_by': '-End Time',
    '_only': ['Machine', 'End Time', 'output'],
}
df = cli.get_cycles(**query)
print(f'Shape: {df.shape}')
df.head()

_limit not specified. Maximum of 5000 rows will be returned.
Shape: (0, 0)


""


## Querying downtimes

`get_downtimes` takes the same query operators as `get_cycles`.

In [19]:
query = {
    'Machine': machines[0],
    'End Time__gte': start_time,
    'End Time__lte': end_time,
    '_order_by': '-End Time',
}
df = cli.get_downtimes(**query)
print(f'Shape: {df.shape}')
df.head()

_limit not specified. Maximum of 5000 rows will be returned.
Shape: (0, 0)


""


### Downtime Pareto — top reasons by total duration
Simple grouping example showing how to turn a downtime query into a quick summary.

In [20]:
if not df.empty and 'Downtime Reason' in df.columns and 'Duration' in df.columns:
    pareto = (
        df.groupby('Downtime Reason')['Duration']
        .sum()
        .sort_values(ascending=False)
        .head(10)
    )
    display(pareto)
else:
    print('No downtime rows / expected columns missing — adjust the date range above.')

No downtime rows / expected columns missing — adjust the date range above.


## Querying parts

Parts track an object across multiple machines. The lookup flow is part type → part data — one step simpler than cycles. Query operators are the same.

In [21]:
# part_types = cli.get_part_type_names()
# part_types

Pull a tiny sample first to discover the available column display names for this part type. (`cli.get_part_schema(part_type)` gives the full field list, but has a pandas-2 compat bug as of April 2026 — sampling works as a substitute.)

In [22]:
# part_type = part_types[0]
# sample = cli.get_parts(Part=part_type, _limit=1)
# part_columns = sample.columns.to_list()
# part_columns[:10]

In [23]:
# query = {
#     'Part': part_type,
#     'End Time__gte': start_time,
#     'End Time__lte': end_time,
#     '_limit': 10
# }
# df = cli.get_parts(**query)
# print(f'Shape: {df.shape}')
# df.head()

## Querying raw data

`get_raw_data` pulls from a raw-data table (pre-cycle sensor stream). You must know the raw-data table name and the field names you want; the API returns a list of records.

See [docs/README.md — Raw Data](../docs/README.md#raw-data) for the full parameter list.

In [24]:
# Replace with a real raw-data table / field names from your tenant.
# time_selection uses the same format as KPI data-viz queries (see notebook 2).
time_selection = {
    'time_type': 'absolute',
    'start_time': start_time.isoformat() + 'Z',
    'end_time': end_time.isoformat() + 'Z',
    'time_zone': 'America/Los_Angeles',
}

# raw_records = cli.get_raw_data(
#     raw_data_table='<raw_table_name>',
#     fields=['<field_a>', '<field_b>'],
#     time_selection=time_selection,
#     limit=100,
# )
# pd.DataFrame(raw_records).head()